<div style="border-left: 4px solid #27ae60; padding-left: 15px; margin-bottom: 20px;">
  <h1 style="margin-top: 0;">🏭⚡ 03. Exploratory Data Analysis: Net-Load Dynamics & Prescriptive Landscape</h1>
  <p style="font-size: 1.1em; color: #888;">Bringing it all together: comparing machine demand, solar generation, electricity prices and estimated CO₂ to explore better scheduling.</p>
</div>

### 🎯 Main Goals & Business Value
This notebook combines measured machine demand, solar output and market prices. We explore what could happen if these systems shared an electrical connection. This is a scheduling scenario, not a verified model of the whole factory.

* **⚖️ Grid Power Needed:** Estimate how much electricity the measured machine would need from the grid after using available solar generation.
* **💶 The Cost Baseline:** Calculate the wholesale import-energy cost over usable shared intervals. This estimate excludes network charges, taxes, demand charges and export payments.
* **🌍 Sustainability:** Estimate grid-import CO₂ using UBA’s published 2024 annual factor of **0.363 kg CO₂/kWh**. Compare how different timing changes this estimate. An annual average cannot identify cleaner hours or prove actual avoided emissions.
* **🧪 The Hindsight Simulation:** Shift the same daily demand profile while keeping solar and prices fixed. Compare costs, grid imports and estimated CO₂ without changing total demand energy. These results are examples, not proven savings or annual ROI.

### 🗺️ Master Blueprint Alignment
This analysis prepares questions and checks for the upcoming `model_training` notebooks. It does not create a new production-ready feature store.

1. **Forecasting:** `01_energy_forecasting_benchmarks.ipynb` will use the ETL-produced model datasets and chronological splits. A 24-hour forecast needs a clearly defined target and evaluation method. Forecasting joint net load also requires confirming that the machine and solar system share an electrical connection.
2. **Anomaly & Degradation:** `02_anomaly_and_degradation.ipynb` will investigate unusual patterns in verified sensor readings. Identifying faults or wear still requires operating context and engineering evidence.
3. **Prescriptive Scheduling & Simulation:** `03_milp_prescriptive_scheduling.ipynb` will develop the scheduling experiment further. A practical optimizer needs job durations, deadlines, staffing, safe starts and other confirmed operating constraints. The annual UBA factor supports estimated carbon comparisons; hourly carbon optimization needs hourly emissions data.

### 🛠️ Core Tools & Approach
* **Data Engineering:** `Polars` aligns identical UTC timestamps from the two 15-minute ETL datasets. Missing readings are not filled or carried into later dates.
* **Simulation:** `NumPy` shifts the daily demand profile while preserving its energy and cyclic order. Grid imports and costs are recalculated for every tested shift.
* **Sustainability:** UBA’s annual factor converts scenario grid-import energy into estimated CO₂. The notebook records the source, publication date and applicable year, and rejects data from other years.
* **Visualization:** `Plotly` presents coverage, energy, costs and estimated carbon comparisons, with short explanations beneath each chart.
* **Pipeline Handoff:** Existing ETL exports remain the source for model training. This notebook does not overwrite them or export an extra unified dataset.

---

### 🗄️ Data Structure
This notebook reads `eda_15min.parquet` for `TEC_48S` and `IPE_PV`. It combines verified total machine power, AC solar generation and matching electricity prices.

The overview cards show the original file sizes, column counts and usable shared coverage. The baseline uses all usable intervals. Daily simulations require complete Berlin calendar days with positive machine demand.

| Domain | Feature Examples | Purpose |
| :--- | :--- | :--- |
| 🏭 **Machine Demand** | `machine_kw`, `machine_kwh` | Measured total machine power and energy per 15-minute interval. |
| ☀️ **Solar Supply** | `solar_kw`, `solar_used_kwh`, `surplus_kwh` | Solar generation, the portion allocated to the machine, and surplus relative to this machine. Surplus is not measured grid export. |
| 🔌 **Grid Power Needed** | `import_kw`, `import_kwh` | Scenario grid demand after subtracting available solar. Imports cannot be negative. |
| 💶 **Market Costs** | `price_eur_mwh`, `import_cost_eur` | Wholesale import cost: import kW × 0.25 hours × EUR/MWh ÷ 1000. Negative prices are retained. |
| 🌍 **Estimated Emissions** | `estimated_import_co2_kg` | Scenario grid imports × 0.363 kg CO₂/kWh using the pinned UBA 2024 factor. |
| ⏱️ **Time & Coverage** | `WsDateTime`, `Date`, `usable` | Exact UTC alignment, Berlin calendar dates and identification of usable shared intervals. |

### 🌍 Emissions Source & Limits
The [UBA publication dated 9 April 2025](https://www.umweltbundesamt.de/themen/co2-emissionen-pro-kilowattstunde-strom-2024) estimates **363 g CO₂/kWh for 2024**. This notebook uses that publication’s annual factor.

The result is a generation-related CO₂ estimate, not measured emissions or lifecycle CO₂-equivalent. UBA notes that emissions associated with net electricity imports arise in other reporting countries. Solar lifecycle emissions and other factory loads are outside this scenario.

Because the factor is constant, lower grid imports produce lower estimated CO₂. It does not show which hour has cleaner electricity. Estimates cover usable shared readings, not a complete annual footprint.

> **Next Step Alignment:** We now have measured demand patterns, a wholesale cost estimate, a UBA-based CO₂ estimate and an energy-preserving scheduling experiment. Model training will use the existing ETL datasets and chronological validation. Predicted grid-import energy can be converted into estimated CO₂ using a year-matched factor. Practical scheduling still requires confirmed operating constraints and electrical connections; hourly carbon optimization requires hourly emissions data.

### 🛠️ 1. Setup and scope
This is a shared-connection scenario for one machine and one solar system. UBA estimates use the shared `features/emissions.py` module. Annual factors do not identify cleaner hours or verify the electrical connection.


In [1]:
import os
os.environ.setdefault("POLARS_MAX_THREADS", "4")
from pathlib import Path
from html import escape
import sys, json, logging
import numpy as np
import polars as pl
import pyarrow.parquet as pq
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown, HTML

TIMEZONE = "Europe/Berlin"
BLUE, PURPLE, ORANGE = "#2563eb", "#7c3aed", "#f59e0b"

def polish(fig, title, subtitle, height=430):
    fig.update_layout(
        template="plotly_white", height=height,
        title=dict(text=f"{title}<br><sup>{subtitle}</sup>", x=0.02, font=dict(size=20)),
        font=dict(family="Arial", size=13, color="#334155"),
        margin=dict(l=70, r=85, t=105, b=65),
        paper_bgcolor="white", plot_bgcolor="white",
        legend=dict(orientation="h", yanchor="bottom", y=1.01, x=0),
        hoverlabel=dict(bgcolor="white"), bargap=0.3,
    )
    fig.update_xaxes(showgrid=False, automargin=True)
    fig.update_yaxes(gridcolor="#eef2f7", automargin=True)
    fig.update_traces(cliponaxis=False, selector=dict(type="bar"))
    if any(t.type == "bar" for t in fig.data) and fig.layout.yaxis.range is None:
        values=[float(v) for t in fig.data if t.type == "bar" for v in t.y if v is not None]
        if values:
            upper = max(values)
            if fig.layout.barmode == "stack":
                upper = sum(max(float(v) for v in t.y) for t in fig.data if t.type == "bar")
            fig.update_yaxes(range=[min(0, min(values)), max(upper * 1.2, 1e-6)])
    fig.show(config={"displaylogo":False,"responsive":True})

def report(text):
    display(Markdown("> **💡 What this tells us**\n> \n" + "\n".join("> " + line for line in text.splitlines())))

def clock(series):
    return series.dt.convert_time_zone(TIMEZONE).dt.strftime("%Y-%m-%d %H:%M:%S").to_list()


def display_cards(cards):
    content = "".join(
        '<div style="padding:20px;border:1px solid #e2e8f0;border-radius:12px;'
        'background:white;color:#334155;min-width:0">'
        f'<div style="font-size:13px;color:#64748b">{escape(label)}</div>'
        f'<div style="font-size:24px;font-weight:700;margin:10px 0">{escape(value)}</div>'
        f'<div style="font-size:12px;line-height:1.5">{escape(note)}</div></div>'
        for label, value, note in cards
    )
    display(HTML('<div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));'
                 'gap:14px;margin:14px 0">' + content + '</div>'))

### 📥 2. Align the data and keep the gaps visible
Read the two ETL-produced 15-minute datasets. Use verified total machine power and AC solar power, both converted to kilowatts by the ETL. Join **identical UTC interval starts**; never carry the last solar reading into later dates. Only finite, non-negative power with matching prices enters the load scenario. Negative machine readings are kept in the source files but excluded here because this scenario treats the machine as a consumer.

In [2]:
project = next((p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / "etl_config.json").is_file()), None)
if project is None:
    raise FileNotFoundError("Run this notebook inside the GreenForge AI project.")
if str(project / "src") not in sys.path:
    sys.path.insert(0, str(project / "src"))
from greenforge_ai.features.interaction_features import artifact
from greenforge_ai.utils.logger import setup_logging, get_logger, stage
setup_logging(project / "logs", new_run=True)
log = get_logger("plant_eda", channel="application")
log.propagate = False
log.handlers = [h for h in logging.getLogger("greenforge.application").handlers
                if isinstance(h, logging.FileHandler)]
paths, quality = {}, {}
for system, target in [("TEC_48S", "P_total"), ("IPE_PV", "AC_ActivePower")]:
    paths[system] = artifact(system, "eda_15min.parquet", project)
    quality[system] = json.loads(artifact(system, "quality.json", project).read_text())
    rule = quality[system].get("rules", {}).get(target, {})
    if rule.get("unit") != "W" or rule.get("status", "unverified").startswith("unverified"):
        raise ValueError(f"Check ETL units for {system}/{target}.")

def read_intervals(system, prefix):
    frame = pl.read_parquet(paths[system], columns=[
        "WsDateTime", "power_kw", "DayAhead_Price_EUR_MWh"
    ]).sort("WsDateTime")
    if frame.is_empty() or frame["WsDateTime"].null_count():
        raise ValueError(f"{system}: empty data or missing timestamps.")
    if not isinstance(frame["WsDateTime"].dtype, pl.Datetime) or frame["WsDateTime"].dtype.time_zone != "UTC":
        raise ValueError(f"{system}: UTC timestamps are required.")
    if frame["WsDateTime"].n_unique() != frame.height:
        raise ValueError(f"{system}: duplicate timestamps.")
    frame = frame.with_columns(pl.col("WsDateTime").cast(pl.Datetime("us", "UTC")))
    if (frame["WsDateTime"].dt.epoch("us") % 900_000_000 != 0).any():
        raise ValueError(f"{system}: interval starts are not on the quarter-hour grid.")
    steps = frame["WsDateTime"].diff().dt.total_microseconds().drop_nulls()
    if (steps != 900_000_000).any():
        raise ValueError(f"{system}: inspect the 15-minute clock.")
    return frame.rename({"power_kw": prefix + "_kw",
                         "DayAhead_Price_EUR_MWh": prefix + "_price"})

with stage("exact interval join and scenario checks", logger=log):
    machine, solar = read_intervals("TEC_48S", "machine"), read_intervals("IPE_PV", "solar")
    grid = machine.join(solar, on="WsDateTime", how="full", coalesce=True,
                        validate="1:1").sort("WsDateTime")
    grid = grid.with_columns(
        pl.col("WsDateTime").dt.convert_time_zone(TIMEZONE).dt.date().alias("Date"),
        pl.col("WsDateTime").dt.convert_time_zone(TIMEZONE).dt.hour().alias("Hour"),
        pl.all_horizontal([pl.col(c).is_finite().fill_null(False)
                           for c in ("machine_kw", "solar_kw", "machine_price", "solar_price")])
        .alias("finite"),
    ).with_columns(
        ((pl.col("machine_price") - pl.col("solar_price")).abs() <= 0.0001)
        .fill_null(False).alias("prices_match"),
    ).with_columns(
        (pl.col("finite") & pl.col("prices_match")
         & (pl.col("machine_kw") >= 0) & (pl.col("solar_kw") >= 0))
        .fill_null(False).alias("usable")
    )
    usable = grid.filter(pl.col("usable")).with_columns(
        pl.col("machine_price").alias("price_eur_mwh"),
        (pl.col("machine_kw") * 0.25).alias("machine_kwh"),
        (pl.col("solar_kw") * 0.25).alias("solar_kwh"),
        pl.min_horizontal("machine_kw", "solar_kw").alias("solar_used_kw"),
        (pl.col("machine_kw") - pl.col("solar_kw")).clip(lower_bound=0).alias("import_kw"),
        (pl.col("solar_kw") - pl.col("machine_kw")).clip(lower_bound=0).alias("surplus_kw"),
    ).with_columns(
        (pl.col("import_kw") * 0.25).alias("import_kwh"),
        (pl.col("solar_used_kw") * 0.25).alias("solar_used_kwh"),
        (pl.col("surplus_kw") * 0.25).alias("surplus_kwh"),
    ).with_columns(
        (pl.col("import_kwh") * pl.col("price_eur_mwh") / 1000).alias("import_cost_eur")
    )
if usable.is_empty():
    raise ValueError("No usable shared intervals; inspect ETL coverage and price alignment.")
if not np.isclose(usable["machine_kwh"].sum(), usable["solar_used_kwh"].sum() + usable["import_kwh"].sum()):
    raise ValueError("Scenario energy balance failed.")
price_conflicts = grid.filter(pl.col("finite") & ~pl.col("prices_match")).height
usable_pct = 100 * usable.height / grid.height
first, last = usable["Date"].min(), usable["Date"].max()
cards = [
    ("Aligned timeline", f"{grid.height:,}", "15-minute interval starts · full outer join"),
    ("Usable shared intervals", f"{usable_pct:.1f}%", f"{usable.height:,} intervals · gaps remain excluded"),
    ("Usable dates", f"{first:%d %b} – {last:%d %b %Y}", "first to last usable reading · gaps may occur within"),
    ("Dataset columns", f"{len(pq.read_schema(paths['TEC_48S']).names)} / {len(pq.read_schema(paths['IPE_PV']).names)}", "machine / solar · original 15-minute files"),
    ("Imported file sizes", f"{paths['TEC_48S'].stat().st_size / 1024**2:.2f} / {paths['IPE_PV'].stat().st_size / 1024**2:.2f} MiB", "machine / solar · compressed on disk"),
    ("Analysis status", "Scenario only", f"{price_conflicts:,} conflicting price intervals excluded · connection unconfirmed"),
]
display_cards(cards)
log.info("Shared intervals=%d/%d price_conflicts=%d", usable.height, grid.height, price_conflicts)

### ⚖️ 3. What would the machine need from the grid?
The first chart shows when the scenario has usable data. The energy chart uses those same intervals for every total. Solar is allocated to the measured machine first; any remaining solar is labelled **surplus relative to this machine**, not measured grid export.

In [3]:
coverage = (
    grid.with_columns(pl.col("Date").dt.truncate("1mo").alias("Month"))
    .group_by("Month")
    .agg(pl.len().alias("intervals"),
         pl.col("usable").sum().alias("usable_intervals"))
    .sort("Month")
    .with_columns(
        (100 * pl.col("usable_intervals") / pl.col("intervals")).alias("coverage_pct")
    )
)

observed = coverage.filter(pl.col("usable_intervals") > 0)
if observed.is_empty():
    raise ValueError("No usable shared intervals.")

shown = coverage.filter(
    pl.col("Month").is_between(observed["Month"].min(), observed["Month"].max())
)

fig = go.Figure(go.Bar(
    x=shown["Month"].dt.strftime("%b %Y").to_list(),
    y=shown["coverage_pct"].to_list(),
    marker_color=BLUE,
    text=[f"{v:.1f}%" for v in shown["coverage_pct"]],
    textposition="outside",
))
fig.update_yaxes(title="Usable shared intervals (%)", range=[0, 110])
fig.update_layout(showlegend=False)
polish(
    fig, "When can machine demand and solar be compared?",
    "Shared-data period only · percentages use the full monthly timeline",
)

report(
    f"**{usable_pct:.1f}% of the full aligned timeline** supports this comparison.\n\n"
    "Months outside the usable shared period are hidden from the chart. "
    "Missing solar readings remain missing; they are not zero generation."
)

> **💡 What this tells us**
> 
> **18.8% of the full aligned timeline** supports this comparison.
> 
> Months outside the usable shared period are hidden from the chart. Missing solar readings remain missing; they are not zero generation.

In [4]:
demand_kwh = float(usable["machine_kwh"].sum())
solar_used_kwh = float(usable["solar_used_kwh"].sum())
import_kwh = float(usable["import_kwh"].sum())
surplus_kwh = float(usable["surplus_kwh"].sum())
fig = go.Figure()
for label, value, color in [("Solar allocated to machine", solar_used_kwh, ORANGE),
                            ("Scenario grid import", import_kwh, BLUE)]:
    fig.add_bar(x=["Measured machine demand"], y=[value], name=label,
                marker_color=color, text=[f"{value:.2f} kWh"], textposition="auto")
fig.update_layout(barmode="stack")
fig.update_yaxes(title="Energy over usable intervals (kWh)", rangemode="tozero")
polish(fig, "Where would the machine's energy come from?", "Shared-connection scenario · no other factory loads included")
report(f"""Across usable intervals, demand is **{demand_kwh:.2f} kWh**: **{solar_used_kwh:.2f} kWh** supplied by solar and **{import_kwh:.2f} kWh** from the grid.
Solar surplus relative to this machine is **{surplus_kwh:.2f} kWh**. It could serve other loads; no export revenue is assumed.""")


> **💡 What this tells us**
> 
> Across usable intervals, demand is **36.13 kWh**: **7.05 kWh** supplied by solar and **29.08 kWh** from the grid.
> Solar surplus relative to this machine is **662.32 kWh**. It could serve other loads; no export revenue is assumed.

### 💶 4. What is the cost baseline?
Each interval costs **import kW × 0.25 hours × EUR/MWh ÷ 1000**. The result is a wholesale import-energy estimate, not the factory bill. It excludes taxes, network charges, demand charges and solar export payments. Negative prices are retained. Daily comparisons use only complete local days, including clock-change days.

In [5]:
with stage("cost baseline and complete local days", logger=log):
    midnight = pl.col("Date").cast(pl.Datetime).dt.replace_time_zone(TIMEZONE).dt.convert_time_zone("UTC")
    next_midnight = (pl.col("Date") + pl.duration(days=1)).cast(pl.Datetime).dt.replace_time_zone(TIMEZONE).dt.convert_time_zone("UTC")
    complete_dates = (
        grid.group_by("Date").agg(pl.len().alias("intervals"), pl.col("usable").sum().alias("valid"))
        .with_columns(((next_midnight - midnight).dt.total_minutes() / 15).alias("expected"))
        .filter((pl.col("intervals") == pl.col("expected")) & (pl.col("valid") == pl.col("intervals")))
        .select("Date")
    )
    daily_cost = usable.group_by("Date").agg(
        pl.col("import_cost_eur").sum().alias("cost_eur"),
        pl.col("machine_kwh").sum(), pl.col("import_kwh").sum(),
    ).join(complete_dates, on="Date", how="inner").sort("Date")
    complete = usable.join(complete_dates, on="Date", how="inner")
    baseline_eur = float(usable["import_cost_eur"].sum())
    retained_kwh = float(complete["machine_kwh"].sum())
    retained_pct = 100 * retained_kwh / demand_kwh if demand_kwh else 0.0

display_cards([
    ("Wholesale baseline", f"€{baseline_eur:.4f}", "all usable intervals · excludes tariff extras"),
    ("Observed time", f"{usable.height * 0.25:,.2f} h", "all usable intervals · not a full-year estimate"),
    ("Complete shared days", f"{daily_cost.height:,}", "92 / 96 / 100 intervals on Berlin clock-change days"),
    ("Demand retained", f"{retained_pct:.1f}%", "share of usable demand on complete days"),
    ("Complete-day demand", f"{retained_kwh:.3f} kWh", "eligible subset for daily simulations"),
    ("Complete-day imports", f"{complete['import_kwh'].sum():.3f} kWh", "solar allocation assumed · same subset"),
])
report("The baseline includes **all usable intervals**. Complete days are used only for daily comparisons and simulations. "
       f"They retain **{retained_pct:.1f}%** of usable machine energy, so their results may not represent the whole timeline.")

> **💡 What this tells us**
> 
> The baseline includes **all usable intervals**. Complete days are used only for daily comparisons and simulations. They retain **5.0%** of usable machine energy, so their results may not represent the whole timeline.

### 🔎 5. One complete day: demand, solar and price
Select the complete positive-demand day with the highest import cost; ties use larger demand, then the earlier date. Both power lines cover the same full local day at 15-minute resolution. Red is machine demand, orange is solar, and dashed blue is price on its own scale. This selection is not the annual machine peak.

In [10]:
candidates = daily_cost.filter(pl.col("machine_kwh") > 0)
example_date, selected = None, None
if candidates.height:
    example_date = candidates.sort(["cost_eur", "machine_kwh", "Date"], descending=[True, True, False])["Date"][0]
    selected = usable.filter(pl.col("Date") == example_date).sort("WsDateTime")
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for column, name, color, secondary in [
        ("machine_kw", "Machine demand", "#dc2626", False),
        ("solar_kw", "Solar power", ORANGE, False),
        ("price_eur_mwh", "Electricity price", BLUE, True),
    ]:
        fig.add_trace(go.Scatter(x=clock(selected["WsDateTime"]), y=selected[column].to_list(), name=name,
            line=dict(color=color, width=2.5, shape="hv", dash="dot" if secondary else "solid"),
            connectgaps=False), secondary_y=secondary)
    fig.update_xaxes(title="Berlin local time", tickformat="%H:%M")
    fig.update_yaxes(title="Power (kW)", secondary_y=False, rangemode="tozero")
    fig.update_yaxes(title="Price (EUR/MWh)", secondary_y=True, showgrid=False)
    fig.update_layout(hovermode="x unified")
    polish(fig, "Demand, solar and price on " + example_date.strftime("%d %B %Y"),
           "Complete 15-minute intervals · largest baseline cost among eligible days", height=480)
    report(f"""This day uses **{selected['machine_kwh'].sum():.2f} kWh** and imports **{selected['import_kwh'].sum():.2f} kWh** in the shared-connection scenario.
The right axis uses a different unit; line crossings do not represent equality.""")
else:
    report("""No complete shared day has positive measured machine energy. A full-day scheduling comparison is unavailable.""")


> **💡 What this tells us**
> 
> This day uses **1.04 kWh** and imports **0.53 kWh** in the shared-connection scenario.
> The right axis uses a different unit; line crossings do not represent equality.

### 🧪 6. Would moving the same demand profile help?
Test every 15-minute shift of the **whole daily demand profile**, keeping solar and prices fixed. The energy and cyclic order of the demand readings stay unchanged. Grid import is recalculated for every shift. This is a hindsight experiment: wrapping across midnight, moving every load, knowing future prices and ignoring staffing or start limits are assumptions, not a feasible production schedule.

In [7]:
savings_eur, import_reduction_kwh = None, None
simulation = None
if selected is not None:
    demand = selected["machine_kw"].to_numpy()
    supply = selected["solar_kw"].to_numpy()
    prices = selected["price_eur_mwh"].to_numpy()
    results = []
    with stage("energy-preserving daily shift scenario", logger=log):
        for shift in range(len(demand)):
            shifted = np.roll(demand, shift)
            imports = np.maximum(shifted - supply, 0) * 0.25
            results.append((shift, float(np.dot(imports, prices) / 1000), float(imports.sum())))
    simulation = pl.DataFrame(results, schema=["shift", "cost_eur", "import_kwh"], orient="row")
    best_cost = simulation.sort(["cost_eur", "shift"]).row(0, named=True)
    best_import = simulation.sort(["import_kwh", "shift"]).row(0, named=True)
    original = results[0]
    savings_eur = original[1] - best_cost["cost_eur"]
    import_reduction_kwh = original[2] - best_import["import_kwh"]
    fig = go.Figure(go.Scatter(x=(simulation["shift"] * 0.25).to_list(),
        y=simulation["cost_eur"].to_list(), mode="lines", line=dict(color=BLUE, width=2.5)))
    fig.add_scatter(x=[0, best_cost["shift"] * 0.25], y=[original[1], best_cost["cost_eur"]],
        mode="markers", marker=dict(color=["#dc2626", PURPLE], size=10), name="Observed / cheapest tested")
    fig.update_xaxes(title="Cyclic shift of demand (elapsed hours)")
    fig.update_yaxes(title="Daily wholesale import cost (EUR)")
    polish(fig, "Cost of moving the same measured demand profile", "Solar and prices fixed · every 15-minute cyclic shift tested")
    report(f"""The original cost is **€{original[1]:.4f}**; the cheapest tested shift costs **€{best_cost['cost_eur']:.4f}**, a difference of **€{savings_eur:.4f}**.
This is a selected-day scenario, not an AI result or proven annual ROI. The best shift is **{best_cost['shift'] * 0.25:.2f} elapsed hours**; feasibility still needs production constraints.""")


> **💡 What this tells us**
> 
> The original cost is **€0.0369**; the cheapest tested shift costs **€0.0140**, a difference of **€0.0229**.
> This is a selected-day scenario, not an AI result or proven annual ROI. The best shift is **2.50 elapsed hours**; feasibility still needs production constraints.

### 🌍 7. Sustainability with a free UBA annual factor
**Estimated CO₂ = scenario grid imports (kWh) × 0.363 kg CO₂/kWh.**

The [Umweltbundesamt publication dated 9 April 2025](https://www.umweltbundesamt.de/themen/co2-emissionen-pro-kilowattstunde-strom-2024) estimates **363 g CO₂/kWh for 2024**. We pin this publication and apply it only to Berlin calendar year 2024. This is an annual generation-related CO₂ estimate, not lifecycle CO₂-equivalent or an hourly carbon signal. UBA notes that emissions associated with net electricity imports arise in other reporting countries.

A fixed annual factor makes lower grid imports equivalent to lower estimated CO₂. It cannot tell us which hour has cleaner electricity or prove actual avoided emissions. Solar lifecycle emissions and other factory loads are outside this scenario.

In [8]:
from greenforge_ai.features.emissions import add_emissions, UBA_FACTORS_KG_PER_KWH, UBA_METADATA

with stage("UBA annual CO2 estimate", logger=log):
    usable = add_emissions(usable.drop("estimated_import_co2_kg", strict=False), logger=log)
    daily_cost = add_emissions(daily_cost.drop("estimated_import_co2_kg", strict=False), logger=log)
    factor = UBA_FACTORS_KG_PER_KWH[2024]
    baseline_co2_kg = float(usable["estimated_import_co2_kg"].sum())
    co2_reduction_kg = None
    log.info("UBA provenance: %s", UBA_METADATA[2024])

report(f"Across **all usable intervals**, scenario imports correspond to **{baseline_co2_kg:.3f} kg estimated CO₂**. "
       "This covers the available shared readings, not a full-year footprint.")
if simulation is not None:
    simulation = simulation.with_columns((pl.col("import_kwh") * factor).alias("estimated_import_co2_kg"))
    labels = ["Observed timing", "Cheapest tested shift", "Least-import shift"]
    imports = [original[2], best_cost["import_kwh"], best_import["import_kwh"]]
    emissions = [value * factor for value in imports]
    co2_reduction_kg = import_reduction_kwh * factor
    fig = go.Figure(go.Bar(x=labels, y=emissions, customdata=imports,
        marker_color=[BLUE, PURPLE, ORANGE], text=[f"{v:.3f} kg" for v in emissions], textposition="outside",
        hovertemplate="%{x}<br>Estimated CO₂: %{y:.3f} kg<br>Grid imports: %{customdata:.3f} kWh<extra></extra>"))
    fig.update_yaxes(title="Estimated grid-import CO₂ (kg)", rangemode="tozero")
    fig.update_layout(showlegend=False)
    polish(fig, "Selected-day cost and carbon choices", "UBA 2024 annual factor · same daily demand · scenario estimates")
    report(f"The least-import shift lowers this annual-factor estimate by **{co2_reduction_kg:.3f} kg CO₂** on the selected day. "
           "The cheapest timing may differ. Hourly emissions and operational feasibility remain unknown.")
else:
    report("No complete positive-demand day is available for a timing comparison. The baseline estimate above still uses all usable intervals.")

> **💡 What this tells us**
> 
> Across **all usable intervals**, scenario imports correspond to **10.557 kg estimated CO₂**. This covers the available shared readings, not a full-year footprint.

> **💡 What this tells us**
> 
> The least-import shift lowers this annual-factor estimate by **0.123 kg CO₂** on the selected day. The cheapest timing may differ. Hourly emissions and operational feasibility remain unknown.

### 🏁 8. Answers and handoff to model training
No extra unified dataset is exported from this notebook. The ETL remains responsible for validated model datasets. Forecasting, anomaly detection and scheduling will need different targets and checks; a neat EDA chart is not a readiness certificate.

In [9]:
ml_notes = []
for system in ("TEC_48S", "IPE_PV"):
    features = quality[system].get("features", {})
    ml_status = features.get("ml_status", "not recorded")
    if ml_status == "ready":
        ml_path = artifact(system, "ml.parquet", project)
        count = pq.read_metadata(ml_path).num_rows
        ml_notes.append(f"**{system}:** ETL reports ready, with **{count:,} exported model windows**.")
    else:
        ml_notes.append(f"**{system}:** ETL model status is **{ml_status}**; inspect the report before training.")
display(Markdown(f"""
**⚖️ What would be bought from the grid?**  
Over usable shared intervals, the scenario imports **{import_kwh:.2f} kWh** to serve **{demand_kwh:.2f} kWh** of machine demand. This assumes a shared connection and excludes other factory loads.

**💶 What is the cost baseline?**  
The wholesale import-energy estimate is **€{baseline_eur:.4f}** over usable intervals. It is neither a full-year bill nor the full tariff cost.

**🧪 Can different timing help?**  
{f'The selected-day experiment found a cost difference of **€{savings_eur:.4f}**.' if savings_eur is not None else 'No complete positive-demand day was available for the experiment.'} Demand energy was preserved, solar and prices stayed fixed, and grid imports were recalculated. Job deadlines and safe operating limits are still missing.

**🌍 What about carbon?**  
Using the pinned UBA 2024 annual factor, usable scenario imports correspond to **{baseline_co2_kg:.3f} kg estimated CO₂**. This is not an hourly or full-year footprint. For ML, convert predicted grid-import kWh with a year-matched factor after prediction; a constant annual factor provides no within-year timing information.

**📚 Next: model-training notebooks**  
{'  '+chr(10).join(ml_notes)}
Start with chronological forecasting baselines using the ETL exports and their time splits. Check each target's units, missing values and forecast horizon before fitting. A 24-hour forecast requires an explicit 24-hour target and validation design; the existing exports must not be relabelled. Joint net-load modelling also requires confirming the electrical relationship. Anomaly and scheduling constraints need further engineering evidence.
"""))


**⚖️ What would be bought from the grid?**  
Over usable shared intervals, the scenario imports **29.08 kWh** to serve **36.13 kWh** of machine demand. This assumes a shared connection and excludes other factory loads.

**💶 What is the cost baseline?**  
The wholesale import-energy estimate is **€1.8624** over usable intervals. It is neither a full-year bill nor the full tariff cost.

**🧪 Can different timing help?**  
The selected-day experiment found a cost difference of **€0.0229**. Demand energy was preserved, solar and prices stayed fixed, and grid imports were recalculated. Job deadlines and safe operating limits are still missing.

**🌍 What about carbon?**  
Using the pinned UBA 2024 annual factor, usable scenario imports correspond to **10.557 kg estimated CO₂**. This is not an hourly or full-year footprint. For ML, convert predicted grid-import kWh with a year-matched factor after prediction; a constant annual factor provides no within-year timing information.

**📚 Next: model-training notebooks**  
  **TEC_48S:** ETL reports ready, with **30,647 exported model windows**.
**IPE_PV:** ETL reports ready, with **8,212 exported model windows**.
Start with chronological forecasting baselines using the ETL exports and their time splits. Check each target's units, missing values and forecast horizon before fitting. A 24-hour forecast requires an explicit 24-hour target and validation design; the existing exports must not be relabelled. Joint net-load modelling also requires confirming the electrical relationship. Anomaly and scheduling constraints need further engineering evidence.
